# Analysis On Brazilian E-Commerce Records

**SIN Chun Hung | HKUST Mathematics | CityUHK Biostatistics**

This research analyzes E-commerce records which are composed of nine csv files from Kaggle. 

https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

The following are the scientific questions this research being investigated by mathematical tools GEE & GLMM. 

Perspective 1: Customer Repurchase Behavior

SQ 1.1: Controlling for order price, freight ratio, cross-state status, and review scores, what is the marginal odds ratio of delivery delay on the overall customer population's repurchase rate? Which working correlation structure (Independence, Exchangeable, AR-1) best describes the temporal dependence of customers?

SQ 1.2: How large is the unobserved heterogeneity among customers ($\sigma^2_u$)? Do the subject-specific parameters ($\beta^*$) under the conditional model and the GEE marginal parameters ($\beta$) satisfy the theoretical shrinkage formula $\beta \approx \beta^* / \sqrt{1 + c^2 \sigma^2_u}$?

SQ 1.3: Is the negative impact of freight ratio on repurchases "heterogeneous across individuals"?

Perspective 2: Seller Complaint Counts & Review of Order

SQ 2.1: Controlling for delay delivery, cross-state ratio, and freight ratio, and using total order volume as the exposure (Offset: $\log(\text{Total Orders})$), can the random intercept of seller state (seller_state) fully absorb overdispersion? Is a Poisson GLMM sufficient, or is a Negative Binomial GLMM (NB GLMM) with a dispersion parameter ($\theta$) required?

SQ 2.2: For binary complaint events at the order level ($Y_{ijk} \in \{0, 1\}$), what proportion of extreme negative review occurrences comes from "seller-level individual heterogeneity" versus "seller state-level clusters"?

Before we dive into these questions, data preparation is the first step of any research. Nine csv files from Kaggle have been processed in PostgreSQL and a final product 'analytical_orders_master' is produced at this stage. This notebook will connect to PostgreSQL server and retrieve this dataset for further preprocessing through R language. 

In [1]:
# ==========================================
# Install and load required packages
# ==========================================
# install.packages(c("DBI", "RPostgres", "dplyr", "geepack", "lme4", "MESS", "MuMIn", "broom", "ggplot2",
#                    "lubridate", "readr", "arrow", "glmmTMB", "performance"))

library(DBI)
library(RPostgres)
library(dplyr)
library(tidyr)
library(lubridate)
library(readr)
library(arrow)
library(geepack)   # Core package for GEE (geeglm)
library(MESS)      # GEE model comparison criterion (QIC)
library(lme4)      # Core package for GLMM (glmer)
library(broom)     # Tidy model output
library(ggplot2)
library(glmmTMB)     # Supports flexible Negative Binomial GLMM
library(performance) # Check overdispersion and model diagnostics


Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union



Attaching package: 'lubridate'


The following objects are masked from 'package:base':

    date, intersect, setdiff, union



Attaching package: 'arrow'


The following object is masked from 'package:lubridate':

    duration


The following object is masked from 'package:utils':

    timestamp



Attaching package: 'MESS'


The following object is masked from 'package:geepack':

    QIC


Loading required package: Matrix


Attaching package: 'Matrix'


The following objects are masked from 'package:tidyr':

    expand, pack, unpack




In [2]:
# ==========================================
# Establish PostgreSQL database connection
# ==========================================

db_user <- "postgres"
db_password <- "hung" 
db_host <- "localhost"
db_port <- 5432
db_name <- "postgres"

con <- dbConnect(
  RPostgres::Postgres(),
  dbname = db_name,
  host = db_host,
  port = db_port,
  user = db_user,
  password = db_password
)

# Verify connection success
cat("Database connection established:", dbIsValid(con), "\n")

Database connection established: TRUE 


In [3]:
# ==========================================
# Load 'analytical_orders_master' table
# ==========================================
query <- "
  SELECT * 
  FROM analytical_orders_master 
  ORDER BY customer_unique_id, order_purchase_timestamp;
"

df_orders <- dbGetQuery(con, query)

# Close database connection
dbDisconnect(con)

# Type casting
df_orders <- df_orders %>%
  mutate(
    order_purchase_timestamp = as.POSIXct(order_purchase_timestamp, tz = "UTC"),
    min_review_score = as.integer(min_review_score),
    is_complaint_review = as.integer(is_complaint_review),
    is_cross_state = as.integer(is_cross_state)
  )

cat("Total delivered orders loaded:", format(nrow(df_orders), big.mark = ","), "\n")

# head(df_orders, n = 3);
# str(df_orders)

Total delivered orders loaded: 96,470 


In [4]:
summary(df_orders)

   order_id         customer_unique_id customer_state     seller_state      
 Length:96470       Length:96470       Length:96470       Length:96470      
 Class :character   Class :character   Class :character   Class :character  
 Mode  :character   Mode  :character   Mode  :character   Mode  :character  
                                                                            
                                                                            
                                                                            
                                                                            
 is_cross_state   order_purchase_timestamp      order_status      
 Min.   :0.0000   Min.   :2016-09-15 12:16:38   Length:96470      
 1st Qu.:0.0000   1st Qu.:2017-09-14 08:56:46   Class :character  
 Median :1.0000   Median :2018-01-20 19:34:43   Mode  :character  
 Mean   :0.6404   Mean   :2018-01-01 23:17:43                     
 3rd Qu.:1.0000   3rd Qu.:2018-05-05 18:29:50    

From the above cells, 'analytical_orders_master' is loaded successfully. Next step is further data preprocessing according to the scientific questions. 'df_repeat_panel', 'df_seller_complaints' and 'df_order_complaints' will be created corresponding to SQ1, SQ2.1 and SQ2.2.  

In [5]:
# ==========================================
# Perspective 1: Process longitudinal panel data for customer repurchase (for GEE / GLMM)
# ==========================================
df_panel_repurchase <- df_orders %>%
  arrange(customer_unique_id, order_purchase_timestamp) %>%
  group_by(customer_unique_id) %>%
  mutate(
    # Current order sequence (t = 1, 2, ...)
    order_seq = row_number(),
    # Total order count for the customer within the observation period
    total_cust_orders = n(),
    # Dependent variable Y_it: Whether there are subsequent orders after this purchase (1 = Repurchased, 0 = None/Last order)
    has_subsequent_purchase = if_else(order_seq < total_cust_orders, 1L, 0L),
    # Days since last order (Time interval)
    days_since_prev_order = as.numeric(difftime(order_purchase_timestamp, lag(order_purchase_timestamp), units = "days")),
    days_since_prev_order = if_else(is.na(days_since_prev_order), 0, days_since_prev_order)
  ) %>%
  ungroup()

# Evaluate longitudinal trace for repeat customer clusters:
df_repeat_panel <- df_panel_repurchase %>%
  filter(total_cust_orders >= 2) %>%
  drop_na(has_subsequent_purchase, delivery_delay_days, actual_delivery_days, 
          total_order_price, freight_ratio, is_cross_state, min_review_score, customer_unique_id) %>%
  arrange(customer_unique_id) %>%
  mutate(
    # Z-score standardization for continuous variables (Accelerate GEE / GLMM convergence and improve numerical stability)
    delivery_delay_days_z  = as.numeric(scale(delivery_delay_days)),
    actual_delivery_days_z = as.numeric(scale(actual_delivery_days)),
    total_order_price_z    = as.numeric(scale(total_order_price)),
    freight_ratio_z        = as.numeric(scale(freight_ratio)),
    # Convert categorical variables to factors
    customer_unique_id     = factor(customer_unique_id),
    customer_state         = factor(customer_state),
    is_cross_state         = factor(is_cross_state)
  ) %>%
  select(
    customer_unique_id,
    order_id,
    order_seq,
    has_subsequent_purchase,
    delivery_delay_days,
    delivery_delay_days_z,
    actual_delivery_days_z,
    total_order_price_z,
    customer_state,
    is_cross_state,
    freight_ratio_z,
    min_review_score
  )

cat("Perspective 1 - Panel record count (Repeat customer orders):", nrow(df_repeat_panel), "\n")
cat("Perspective 1 - Unique cluster count (Unique customers):", n_distinct(df_repeat_panel$customer_unique_id), "\n")

Perspective 1 - Panel record count (Repeat customer orders): 5869 
Perspective 1 - Unique cluster count (Unique customers): 2792 


In [6]:
summary(df_repeat_panel)

                        customer_unique_id   order_id           order_seq     
 8d50f5eadf50201ccdcedfb9e2ac8455:  15     Length:5869        Min.   : 1.000  
 3e43e6105506432c953e165fb2acf44c:   9     Class :character   1st Qu.: 1.000  
 1b6c7548a2a1f9037c1fd3ddfed95f33:   7     Mode  :character   Median : 2.000  
 6469f99c1f9dfae7733b25662e7f1782:   7                        Mean   : 1.617  
 ca77025e7201e3b30c44b472ff346268:   7                        3rd Qu.: 2.000  
 12f5d6e1cbf93dafd9dcc19095df0b3d:   6                        Max.   :15.000  
 (Other)                         :5818                                        
 has_subsequent_purchase delivery_delay_days delivery_delay_days_z
 Min.   :0.000           Min.   :-123.433    Min.   :-11.42489    
 1st Qu.:0.000           1st Qu.: -16.714    1st Qu.: -0.48306    
 Median :1.000           Median : -12.224    Median : -0.02271    
 Mean   :0.527           Mean   : -12.002    Mean   :  0.00000    
 3rd Qu.:1.000           3rd Qu.:

In [7]:
# ==========================================
# Perspective 2: Process seller complaint count data (Count Data for Poisson / NB GLMM) SQ 2.1
# ==========================================
df_seller_complaints <- df_orders %>%
  # Retain single-seller orders only
  filter(distinct_sellers_count == 1, !is.na(primary_seller_id)) %>%
  group_by(primary_seller_id) %>%
  summarise(
    total_orders       = n(),
    complaint_count    = sum(is_complaint_review, na.rm = TRUE),
    avg_delay_days     = mean(delivery_delay_days, na.rm = TRUE),
    pct_cross_state    = mean(is_cross_state, na.rm = TRUE),
    avg_freight_ratio  = mean(freight_ratio, na.rm = TRUE),
    seller_state       = first(seller_state),
    .groups            = "drop"
  ) %>%
  # Exclude extremely low-frequency sellers to prevent random intercept instability caused by data sparsity
  filter(total_orders >= 5) %>%
  mutate(
    # Exposure Offset (Required for Poisson / NB GLMM)
    log_total_orders   = log(total_orders),
    # Z-score standardization for continuous predictors
    avg_delay_days_z    = as.numeric(scale(avg_delay_days)),
    pct_cross_state_z   = as.numeric(scale(pct_cross_state)),
    avg_freight_ratio_z = as.numeric(scale(avg_freight_ratio)),
    seller_state        = factor(seller_state),
    primary_seller_id   = factor(primary_seller_id)
  )

cat("Perspective 2 - Analyzed seller count (Sellers):", nrow(df_seller_complaints), "\n")

Perspective 2 - Analyzed seller count (Sellers): 1742 


In [8]:
summary(df_seller_complaints)

                        primary_seller_id  total_orders     complaint_count  
 001cca7ae9ae17fb1caed9dfb1094831:   1    Min.   :   5.00   Min.   :  0.000  
 002100f778ceb8431b7a1020ff7ab48f:   1    1st Qu.:   8.00   1st Qu.:  1.000  
 004c9cd9d87a3c30c522c48c4fc07416:   1    Median :  17.00   Median :  2.000  
 00720abe85ba0859807595bbf045a33b:   1    Mean   :  53.27   Mean   :  6.568  
 00ee68308b45bc5e2660cd833c3f81cc:   1    3rd Qu.:  45.00   3rd Qu.:  5.000  
 00fc707aaaad2d31347cf883cd2dfe10:   1    Max.   :1788.00   Max.   :283.000  
 (Other)                         :1736                                       
 avg_delay_days    pct_cross_state  avg_freight_ratio  seller_state 
 Min.   :-47.549   Min.   :0.0000   Min.   :0.01322   SP     :1069  
 1st Qu.:-13.548   1st Qu.:0.5000   1st Qu.:0.13637   PR     : 189  
 Median :-10.950   Median :0.6364   Median :0.19504   MG     : 143  
 Mean   :-11.346   Mean   :0.6568   Mean   :0.20542   RJ     :  99  
 3rd Qu.: -8.912   3rd Qu.:0.84

In [9]:
# =======================================================
# Process independent order-level complaint analysis dataset SQ 2.2
# =======================================================

# Filter single-seller orders and select mandatory fields for Perspective 2
df_order_complaints <- df_orders %>%
  filter(distinct_sellers_count == 1, !is.na(primary_seller_id)) %>%
  mutate(
    # Binary complaint indicator (Review Score <= 2)
    is_complaint_review = as.integer(is_complaint_review),
    # Factorize categorical variables
    primary_seller_id   = factor(primary_seller_id),
    seller_state        = factor(seller_state),
    customer_state      = factor(customer_state),
    is_cross_state      = factor(is_cross_state),
    # Standardize continuous predictors (Z-score)
    delivery_delay_days_z = as.numeric(scale(delivery_delay_days)),
    freight_ratio_z       = as.numeric(scale(freight_ratio)),
    actual_delivery_days_z = as.numeric(scale(actual_delivery_days))
  ) %>%
  select(
    order_id,
    primary_seller_id,
    seller_state,
    customer_state,
    is_cross_state,
    product_category,
    is_complaint_review,
    delivery_delay_days_z,
    actual_delivery_days_z,
    freight_ratio_z
  )

cat("Perspective 2 - Total order-level observations:", nrow(df_order_complaints), "\n")
cat("Number of sellers (Seller Clusters):", n_distinct(df_order_complaints$primary_seller_id), "\n")
cat("Number of seller states (State Clusters):",  n_distinct(df_order_complaints$seller_state), "\n")

Perspective 2 - Total order-level observations: 95195 
Number of sellers (Seller Clusters): 2948 
Number of seller states (State Clusters): 22 


In [10]:
# ==========================================
# Output cache files (Parquet / RDS formats for easy future loading)
# ==========================================
write_rds(df_repeat_panel, "df_repeat_panel.rds")
write_rds(df_seller_complaints, "df_seller_complaints.rds")
write_rds(df_order_complaints, "df_order_complaints.rds")

# Simultaneously save as cross-language universal Parquet
write_parquet(df_repeat_panel, "df_repeat_panel.parquet")
write_parquet(df_seller_complaints, "df_seller_complaints.parquet")
write_parquet(df_order_complaints, "df_order_complaints.parquet")

cat("Data preprocessing complete. Successfully saved df_repeat_panel, df_seller_complaints and df_order_complaints!\n")

Data preprocessing complete. Successfully saved df_repeat_panel, df_seller_complaints and df_order_complaints!


In [11]:
# Load clean panel data
df_repeat_panel <- read_rds("df_repeat_panel.rds")

# Ensure data is sorted by cluster ID and order sequence (Strict requirement for geepack)
df_repeat_panel <- df_repeat_panel %>%
  arrange(customer_unique_id, order_seq)

cat("Total analysis sample size:", nrow(df_repeat_panel), "Number of clusters:", n_distinct(df_repeat_panel$customer_unique_id), "\n")

Total analysis sample size: 5869 Number of clusters: 2792 


## SQ 1.1: Marginal Effect of Logistics on Population Repurchase Rate

Controlling for order price, freight ratio, cross-state status, and review scores, what is the marginal odds ratio of delivery delay on the overall customer population's repurchase rate? Which working correlation structure (Independence, Exchangeable, AR-1) best describes the temporal dependence of customers?

To discover the logistics marginal effect on customer repurchase behavior, we first fit three GEE model with 3 different working correlation structure followed by QIC technique to choose the best model. 

In [12]:
# =============================================================
# SQ 1.1: GEE Modeling and Working Correlation Structure Evaluation
# =============================================================

# (1) Independence
gee_ind <- geeglm(has_subsequent_purchase ~ delivery_delay_days_z + actual_delivery_days_z + 
                    total_order_price_z + freight_ratio_z + is_cross_state + min_review_score, 
                  data = df_repeat_panel, 
                  id = customer_unique_id, 
                  family = binomial(link = "logit"), 
                  corstr = "independence")

# (2) Exchangeable
gee_exch <- geeglm(has_subsequent_purchase ~ delivery_delay_days_z + actual_delivery_days_z + 
                     total_order_price_z + freight_ratio_z + is_cross_state + min_review_score, 
                   data = df_repeat_panel, 
                   id = customer_unique_id, 
                   family = binomial(link = "logit"), 
                   corstr = "exchangeable")

# (3) AR-1
gee_ar1 <- geeglm(has_subsequent_purchase ~ delivery_delay_days_z + actual_delivery_days_z + 
                    total_order_price_z + freight_ratio_z + is_cross_state + min_review_score, 
                  data = df_repeat_panel, 
                  id = customer_unique_id, 
                  family = binomial(link = "logit"), 
                  corstr = "ar1")

In [13]:
# Model selection: Calculate Quasi-Likelihood under the Independence Model Criterion (QIC)
qic_ind  <- MESS::QIC(gee_ind)
qic_exch <- MESS::QIC(gee_exch)
qic_ar1  <- MESS::QIC(gee_ar1)

qic_comparison <- data.frame(
  Model_Name = c("gee_ind", "gee_exch", "gee_ar1"),
  Correlation_Structure = c("Independence", "Exchangeable", "AR-1"),
  QIC                   = c(qic_ind["QIC"], qic_exch["QIC"], qic_ar1["QIC"]),
  QICu                  = c(qic_ind["QICu"], qic_exch["QICu"], qic_ar1["QICu"])
)
print(qic_comparison)

  Model_Name Correlation_Structure      QIC     QICu
1    gee_ind          Independence 8118.205 8124.463
2   gee_exch          Exchangeable 8168.521 8181.236
3    gee_ar1                  AR-1 8124.512 8134.516


From above result, the QIC of gee_ind is the lowest, it implies we should choose GEE independent working corelation structure to examine the coefficients vector of covariate and its exponential & p-value.  

In [14]:
# Extract marginal odds ratios (Marginal OR) and sandwich robust standard errors (Sandwich SE) from the optimal GEE model
best_idx <- which.min(qic_comparison$QIC)
best_gee <- get(qic_comparison$Model_Name[best_idx])  # In practice, the model with the minimum QIC is preferred
print(qic_comparison$Model_Name[best_idx])
gee_summary <- summary(best_gee)

gee_results <- as.data.frame(gee_summary$coefficients) %>%
  mutate(
    Marginal_OR = exp(Estimate),
    OR_CI_lower = exp(Estimate - 1.96 * `Std.err`),
    OR_CI_upper = exp(Estimate + 1.96 * `Std.err`)
  )

print(round(gee_results[, c("Estimate", "Std.err", "Wald", "Pr(>|W|)", "Marginal_OR", "OR_CI_lower", "OR_CI_upper")], 4))

[1] "gee_ind"
                       Estimate Std.err   Wald Pr(>|W|) Marginal_OR OR_CI_lower
(Intercept)              0.1324  0.0746 3.1471   0.0761      1.1415      0.9862
delivery_delay_days_z   -0.0765  0.0304 6.3145   0.0120      0.9264      0.8727
actual_delivery_days_z   0.0872  0.0318 7.5210   0.0061      1.0911      1.0252
total_order_price_z      0.0210  0.0224 0.8778   0.3488      1.0212      0.9774
freight_ratio_z          0.0333  0.0250 1.7682   0.1836      1.0338      0.9843
is_cross_state1         -0.1001  0.0477 4.4075   0.0358      0.9047      0.8240
min_review_score         0.0093  0.0167 0.3098   0.5778      1.0093      0.9769
                       OR_CI_upper
(Intercept)                 1.3213
delivery_delay_days_z       0.9833
actual_delivery_days_z      1.1612
total_order_price_z         1.0669
freight_ratio_z             1.0858
is_cross_state1             0.9934
min_review_score            1.0429


From above result, the variables 'delivery_delay_days_z', 'actual_delivery_days_z', 'is_cross_state1' p-values are all smaller than 0.05, implies they are significant to GEE model (NOT zero) under 95% confidence level. By this, we conclude in following. 

Adjusting for other factors, the odd of customer repurchase will be decreased by 7.36% of the original with 1 unit increase (or 1 standard deviation increase) in delivery_delay_days_z. This means the longer time the customer receive his/her product, the more likely he/she will not repurchase later time. 

## SQ 1.2: Individual Heterogeneity & Subject-Specific Effect vs. Marginal Effect

How large is the unobserved heterogeneity among customers ($\sigma^2_u$)?

Do the subject-specific parameters ($\beta^*$) under the conditional model and the GEE marginal parameters ($\beta$) satisfy the theoretical shrinkage formula $\beta \approx \beta^* / \sqrt{1 + c^2 \sigma^2_u}$?

Unobserved heterogeneity may exist among the group of customers which cannot be captured by GEE model in SQ1.1, we introduce random intercept model i.e. $b_{i} \sim N(0, \sigma^2_{\text{customer}})$ = random intercept of every customer i to allow variation of intercept of different customer to discover the effect of unobserved heterogeneity.



In [15]:
# =============================================================
# SQ 1.2: GLMM Random Intercept Model Fitting and Theoretical Attenuation Test
# =============================================================

# Fit Logit-link GLMM (Random Intercept Model)
glmm_ri <- glmer(
  has_subsequent_purchase ~ delivery_delay_days_z + actual_delivery_days_z + 
                            total_order_price_z + freight_ratio_z + 
                            is_cross_state + min_review_score + 
                            (1 | customer_unique_id),
  data      = df_repeat_panel,
  family    = binomial(link = "logit"),
  control   = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e5))
)

summary(glmm_ri)

boundary (singular) fit: see help('isSingular')



Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: 
has_subsequent_purchase ~ delivery_delay_days_z + actual_delivery_days_z +  
    total_order_price_z + freight_ratio_z + is_cross_state +  
    min_review_score + (1 | customer_unique_id)
   Data: df_repeat_panel
Control: glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e+05))

      AIC       BIC    logLik -2*log(L)  df.resid 
   8126.5    8179.9   -4055.2    8110.5      5861 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-1.6913 -1.0463  0.8849  0.9533  1.0559 

Random effects:
 Groups             Name        Variance Std.Dev.
 customer_unique_id (Intercept) 0        0       
Number of obs: 5869, groups:  customer_unique_id, 2792

Fixed effects:
                        Estimate Std. Error z value Pr(>|z|)  
(Intercept)             0.132384   0.102296   1.294   0.1956  
delivery_delay_days_z  -0.076496   0.034130  -2.241   0.025

In [16]:
# Extract random effects variance (tau^2) and calculate VPC
var_comp <- as.data.frame(VarCorr(glmm_ri))
sigma_u_sq <- var_comp$vcov[1] # Random intercept variance

# Standard residual variance for the Logit distribution is pi^2 / 3 ≈ 3.289868
vpc <- sigma_u_sq / (sigma_u_sq + (pi^2 / 3))

cat(sprintf("Customer-level unobserved heterogeneity variance (sigma_u^2): %.4f\n", sigma_u_sq))
cat(sprintf("Variance Partition Coefficient (VPC): %.4f\n", vpc))

Customer-level unobserved heterogeneity variance (sigma_u^2): 0.0000
Variance Partition Coefficient (VPC): 0.0000


From above results, the variance partition coefficient is zero due to the variance of random intercept of customer cluster level is zero after controlling the covariates indicated in summary result of glmm. This implies repurchase behavior for each customer in dataset does not vary, the unobserved heterogeneity among customers is zero. 

In [17]:
# Empirical comparison: Subject-Specific Effect vs. Population-Averaged Effect
glmm_fixef <- fixef(glmm_ri)
gee_coef   <- coef(best_gee)

# Theoretical attenuation factor (c ≈ 16 * sqrt(3) / (15 * pi) ≈ 0.588, commonly c^2 ≈ 0.346)
attenuation_factor <- sqrt(1 + 0.346 * sigma_u_sq)

comparison_df <- data.frame(
  Feature              = names(glmm_fixef),
  GLMM_Beta_Conditional = as.numeric(glmm_fixef),
  GLMM_Conditional_OR  = exp(as.numeric(glmm_fixef)),
  GEE_Beta_Marginal    = as.numeric(gee_coef[names(glmm_fixef)]),
  GEE_Marginal_OR      = exp(as.numeric(gee_coef[names(glmm_fixef)])),
  Predicted_Marginal_Beta = as.numeric(glmm_fixef) / attenuation_factor
)

print(round(comparison_df[, -1], 4), row.names = comparison_df$Feature)

                       GLMM_Beta_Conditional GLMM_Conditional_OR
(Intercept)                           0.1324              1.1415
delivery_delay_days_z                -0.0765              0.9264
actual_delivery_days_z                0.0872              1.0911
total_order_price_z                   0.0210              1.0212
freight_ratio_z                       0.0333              1.0338
is_cross_state1                      -0.1001              0.9047
min_review_score                      0.0093              1.0093
                       GEE_Beta_Marginal GEE_Marginal_OR
(Intercept)                       0.1324          1.1415
delivery_delay_days_z            -0.0765          0.9264
actual_delivery_days_z            0.0872          1.0911
total_order_price_z               0.0210          1.0212
freight_ratio_z                   0.0333          1.0338
is_cross_state1                  -0.1001          0.9047
min_review_score                  0.0093          1.0093
                       P

As $\sigma^2_{\text{customer}} = 0$ shows above, $\beta \approx \beta^* / \sqrt{1 + c^2 \sigma^2_u}$ reduces to $\beta = \beta^*$.

This indicates that the coefficients of covariate in GLMM with random intercept is the same as that of GEE model fit in SQ1.1. The interpretation of odd ratio is the same. The slightly difference between this GLMM and GEE in SQ1.1 is that the is_cross_state1 is no longer significant to model as p-value is larger than 0.05 at 95% significant level. 

The reason of the variance of random intercept of cluster-level (customer cluster) is zero maybe due to the samples within each cluster (customer)'s variance is very small or the repeated measurement within cluster is not large enough. 

## SQ 1.3: Individual Differences in Price Sensitivity

Is the negative impact of freight ratio on repurchases "heterogeneous across individuals"?

Below we introduce GLMM with both random slope and random intercept, the random intercept is the same as SQ1.2, the random slope of freight ratio allows the freight effect to vary by customer. 
 
Finally we verify whether this random slope effect is statistically significant through Likelihood Ratio Test (LRT).

In [18]:
# =============================================================
# SQ 1.3: Random Slope Model Extension and Likelihood Ratio Test (LRT)
# =============================================================

# Fit GLMM including a random slope for 'freight_ratio_z'
# (1 + freight_ratio_z | customer_unique_id) to estimate random intercepts, random slopes, and their correlation
glmm_rs <- glmer(
  has_subsequent_purchase ~ delivery_delay_days_z + actual_delivery_days_z + 
                            total_order_price_z + freight_ratio_z + 
                            is_cross_state + min_review_score + 
                            (1 + freight_ratio_z | customer_unique_id),
  data      = df_repeat_panel,
  family    = binomial(link = "logit"),
  control   = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e5))
)

summary(glmm_rs)

boundary (singular) fit: see help('isSingular')



Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: 
has_subsequent_purchase ~ delivery_delay_days_z + actual_delivery_days_z +  
    total_order_price_z + freight_ratio_z + is_cross_state +  
    min_review_score + (1 + freight_ratio_z | customer_unique_id)
   Data: df_repeat_panel
Control: glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e+05))

      AIC       BIC    logLik -2*log(L)  df.resid 
   8130.5    8197.2   -4055.2    8110.5      5859 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-1.6913 -1.0463  0.8849  0.9533  1.0559 

Random effects:
 Groups             Name            Variance  Std.Dev.  Corr 
 customer_unique_id (Intercept)     0.000e+00 0.000e+00      
                    freight_ratio_z 3.896e-19 6.242e-10  NaN 
Number of obs: 5869, groups:  customer_unique_id, 2792

Fixed effects:
                        Estimate Std. Error z value Pr(>|z|)  
(Intercept)       

In [19]:
# Execute Likelihood Ratio Test (LRT) for nested model comparison
# Test the null hypothesis H0: sigma_slope^2 = 0 and cov(intercept, slope) = 0
lrt_test <- anova(glmm_ri, glmm_rs)
print(lrt_test)

Data: df_repeat_panel
Models:
glmm_ri: has_subsequent_purchase ~ delivery_delay_days_z + actual_delivery_days_z + total_order_price_z + freight_ratio_z + is_cross_state + min_review_score + (1 | customer_unique_id)
glmm_rs: has_subsequent_purchase ~ delivery_delay_days_z + actual_delivery_days_z + total_order_price_z + freight_ratio_z + is_cross_state + min_review_score + (1 + freight_ratio_z | customer_unique_id)
        npar    AIC    BIC  logLik -2*log(L) Chisq Df Pr(>Chisq)
glmm_ri    8 8126.5 8179.9 -4055.2    8110.5                    
glmm_rs   10 8130.5 8197.2 -4055.2    8110.5     0  2          1


In [20]:
# # Extract correlation between random intercepts and random slopes (Correlation between Baseline & Freight Sensitivity)
rs_varcorr <- VarCorr(glmm_rs)
corr_matrix <- attr(rs_varcorr$customer_unique_id, "correlation")
cat("Correlation coefficient between random intercept and freight random slope:", round(corr_matrix[1, 2], 4), "\n")

Correlation coefficient between random intercept and freight random slope: NaN 


The LRT for nested model comparison glmm_ri (SQ1.2) and glmm_rs (random slope model SQ1.3) shows that the p-value is equal to 1. This proves that adding random slope does not improve the model fit over the random intercept model and the random slope i.e. freight ratio is not statistically significant and is not heterogeneous across individuals. 

Besides, adding the random slope of freight ratio will impose penalty and increase the AIC and BIC. Therefore, random intercept model for customer is better than random slope introduced to model. 

## SQ 2.1: Seller Cumulative Complaint Count: Overdispersion

Controlling for delay delivery, cross-state ratio, and freight ratio, and using total order volume as the exposure (Offset: $\log(\text{Total Orders})$), can the random intercept of seller state (seller_state) fully absorb overdispersion? 

Is a Poisson GLMM sufficient, or is a Negative Binomial GLMM (NB GLMM) with a dispersion parameter ($\alpha$) required?

In [21]:
# # Load aggregated seller data and order table
df_seller_complaints <- read_rds("df_seller_complaints.rds")
df_repeat_panel      <- read_rds("df_repeat_panel.rds")

cat("Seller count dataset dimensions:", nrow(df_seller_complaints), "seller records\n")

Seller count dataset dimensions: 1742 seller records


In [22]:
# =============================================================
# SQ 2.1: Seller Cumulative Negative Reviews Modeling and Overdispersion Test
# =============================================================

# Baseline Poisson GLMM (with 'seller_state' as random intercept, and log-transformed order volume as offset)
poisson_glmm <- glmer(
  complaint_count ~ avg_delay_days_z + pct_cross_state_z + avg_freight_ratio_z + 
                    offset(log_total_orders) + (1 | seller_state),
  data    = df_seller_complaints,
  family  = poisson(link = "log"),
  control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e5))
)

summary(poisson_glmm)

# Statistical test for overdispersion (Overdispersion Test)
disp_check <- performance::check_overdispersion(poisson_glmm)
print(disp_check)

Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: poisson  ( log )
Formula: 
complaint_count ~ avg_delay_days_z + pct_cross_state_z + avg_freight_ratio_z +  
    offset(log_total_orders) + (1 | seller_state)
   Data: df_seller_complaints
Control: glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e+05))

      AIC       BIC    logLik -2*log(L)  df.resid 
   7550.4    7577.7   -3770.2    7540.4      1737 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-4.2052 -0.9199 -0.2815  0.4901 12.7846 

Random effects:
 Groups       Name        Variance Std.Dev.
 seller_state (Intercept) 0.01393  0.118   
Number of obs: 1742, groups:  seller_state, 20

Fixed effects:
                    Estimate Std. Error z value Pr(>|z|)    
(Intercept)         -2.25938    0.04700 -48.068  < 2e-16 ***
avg_delay_days_z     0.22025    0.01630  13.515  < 2e-16 ***
pct_cross_state_z    0.13534    0.02197   6.161 7.23e-10 ***
avg_freight_rati

# Overdispersion test (using simulated residuals)

 dispersion ratio =   2.854
          p-value = < 0.001



Overdispersion detected.



From above result, the random intercept seller state for each seller in this Poisson GLMM cannot fully absorb overdispersion as the p-value is smaller than 0.001 statistically indicates detection of overdispersion. This shows the disadvantage of Poisson model that assume the variance is the same as expectation/mean. This implies the needs of Negative Binomial GLMM which introduce a dispersion parameter $\alpha$ to absorb dispersion as the formula  $\text{Var}(Y) = \mu + \alpha \mu^2$.

In [23]:
# Fit Negative Binomial GLMM (with log link)
# If Poisson exhibits significant overdispersion, use glmmTMB to estimate the extra dispersion parameter
nb_glmm <- glmmTMB(
  complaint_count ~ avg_delay_days_z + pct_cross_state_z + avg_freight_ratio_z + 
                    offset(log_total_orders) + (1 | seller_state),
  data    = df_seller_complaints,
  family  = nbinom2(link = "log")
)

summary(nb_glmm)

 Family: nbinom2  ( log )
Formula:          
complaint_count ~ avg_delay_days_z + pct_cross_state_z + avg_freight_ratio_z +  
    offset(log_total_orders) + (1 | seller_state)
Data: df_seller_complaints

      AIC       BIC    logLik -2*log(L)  df.resid 
   6910.8    6943.6   -3449.4    6898.8      1736 

Random effects:

Conditional model:
 Groups       Name        Variance Std.Dev.
 seller_state (Intercept) 0.00394  0.06277 
Number of obs: 1742, groups:  seller_state, 20

Dispersion parameter for nbinom2 family (): 7.04 

Conditional model:
                     Estimate Std. Error z value Pr(>|z|)    
(Intercept)         -2.209243   0.047444  -46.57  < 2e-16 ***
avg_delay_days_z     0.205270   0.020851    9.84  < 2e-16 ***
pct_cross_state_z    0.089867   0.034026    2.64  0.00826 ** 
avg_freight_ratio_z  0.005515   0.017484    0.32  0.75241    
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

In [24]:
# Model comparison using AIC
aic_poisson <- AIC(poisson_glmm)
aic_nb      <- AIC(nb_glmm)
cat(sprintf("Poisson GLMM AIC: %.2f | NB GLMM AIC: %.2f\n", aic_poisson, aic_nb))

Poisson GLMM AIC: 7550.39 | NB GLMM AIC: 6910.80


AIC value is dramatically decrease from 7550.39 to 6910.80, switching from Poisson GLMM to Negative Binomial GLMM respectively. This proves that the goodness of fit of latter is much better than that of former.

In [25]:
# Extract Incidence Rate Ratios (IRR) and confidence intervals
nb_coefs <- summary(nb_glmm)$coefficients$cond
irr_table <- data.frame(
  Feature    = rownames(nb_coefs),
  Estimate   = nb_coefs[, "Estimate"],
  Std_Error  = nb_coefs[, "Std. Error"],
  IRR        = exp(nb_coefs[, "Estimate"]),
  CI_2.5pct  = exp(nb_coefs[, "Estimate"] - 1.96 * nb_coefs[, "Std. Error"]),
  CI_97.5pct = exp(nb_coefs[, "Estimate"] + 1.96 * nb_coefs[, "Std. Error"]),
  P_value    = nb_coefs[, "Pr(>|z|)"]
)
print(round(irr_table[, -1], 4), row.names = irr_table$Feature)

                    Estimate Std_Error    IRR CI_2.5pct CI_97.5pct P_value
(Intercept)          -2.2092    0.0474 0.1098    0.1000     0.1205  0.0000
avg_delay_days_z      0.2053    0.0209 1.2279    1.1787     1.2791  0.0000
pct_cross_state_z     0.0899    0.0340 1.0940    1.0234     1.1695  0.0083
avg_freight_ratio_z   0.0055    0.0175 1.0055    0.9717     1.0406  0.7524


The incidence rate ratios table shows that p-value of 'avg_delay_days_z', 'pct_cross_state_z' are smaller than 0.05, they are statistically significant to Negative Binomial GLMM at 95% confidence level. By this, we conclude: 

Conditional on seller state and adjusting other factors, the incidence rate of complaint over total order will dramatically increase 22.8% of the original with 1-unit/1 standard deviation increase in average delay day.

Conditional on seller state and adjusting other factors, the percentage cross-state transaction increase 1 standard deviation will increase complaint incidence rate 9.40% of original, this reflects the complexity of interstate shipping can cause higher complaint incidence rate.  

Compare 'avg_freight_ratio_z' in NB GLMM with Poisson GLMM, the p-value of Poisson GLMM is 0.0522 which is much smaller than that of NB GLMM. This means overdispersion will mistakenly underestimate the standard error causing wrong or fake p-value estimation. After adopting NB GLMM, the standard error is fixed and reflects that average freight ratio in fact have not much contribution to complaint count.

## SQ 2.2: Multi-level Complaint Risk Attribution: Variance Partition Coefficients (VPC)

For binary complaint events at the order level ($Y_{ijk} \in \{0, 1\}$), what proportion of extreme negative review occurrences comes from "seller-level individual heterogeneity" versus "seller state-level clusters"?

Below cell will fit the Multilevel Binomial GLMM:

$\text{logit}(P(Y_{ijk} = 1)) = \beta_0 + \mathbf{X}_{ijk}\boldsymbol{\beta} + u_k + v_{jk}$, where $u_k \sim N(0, \sigma^2_{\text{state}})$ is state random intercept and $v_{jk} \sim N(0, \sigma^2_{\text{seller}})$ is seller random intercept, state is higher level cluster.

In [26]:
# =============================================================
# SQ 2.2: Multilevel Binomial GLMM and VPC Decomposition
# =============================================================

# Fit a dual-random-intercept model: (1 | seller_state) + (1 | primary_seller_id)
# Given the sample size of ~90,000 observations, the 'bobyqa' optimizer is used to ensure stable convergence
glmm_complaint_multilevel <- glmer(
  is_complaint_review ~ delivery_delay_days_z + freight_ratio_z + is_cross_state + 
                        (1 | seller_state) + (1 | primary_seller_id),
  data    = df_order_complaints,
  family  = binomial(link = "logit"),
  control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e5))
)

summary(glmm_complaint_multilevel)

Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: is_complaint_review ~ delivery_delay_days_z + freight_ratio_z +  
    is_cross_state + (1 | seller_state) + (1 | primary_seller_id)
   Data: df_order_complaints
Control: glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 2e+05))

      AIC       BIC    logLik -2*log(L)  df.resid 
  63298.6   63355.4  -31643.3   63286.6     95189 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-984.95   -0.37   -0.29   -0.22   22.47 

Random effects:
 Groups            Name        Variance Std.Dev.
 primary_seller_id (Intercept) 0.2546   0.5045  
 seller_state      (Intercept) 0.0000   0.0000  
Number of obs: 95195, groups:  primary_seller_id, 2948; seller_state, 22

Fixed effects:
                      Estimate Std. Error z value Pr(>|z|)    
(Intercept)           -2.60187    0.02630 -98.912   <2e-16 ***
delivery_delay_days_z  0.81378    0.01137  71.5

In [27]:
# # Extract random effects variance components
var_comp_table <- as.data.frame(VarCorr(glmm_complaint_multilevel))

sigma_seller_sq <- var_comp_table$vcov[var_comp_table$grp == "primary_seller_id"]
sigma_state_sq  <- var_comp_table$vcov[var_comp_table$grp == "seller_state"]
sigma_resid_sq  <- (pi^2) / 3   # Logistic latent variable residual variance ≈ 3.289868

total_latent_variance <- sigma_seller_sq + sigma_state_sq + sigma_resid_sq

In [28]:
# Calculate Variance Partition Coefficients (VPC)
vpc_seller <- sigma_seller_sq / total_latent_variance
vpc_state  <- sigma_state_sq / total_latent_variance
vpc_residual <- sigma_resid_sq / total_latent_variance

cat("\n================ Multilevel Complaint Risk Variance Decomposition (VPC) ================\n")
cat(sprintf("1. Individual seller effect variance  (sigma^2_seller)   : %.4f | %% of Total Variance: %6.2f%%\n", sigma_seller_sq, vpc_seller * 100))
cat(sprintf("2. Seller state geographical variance (sigma^2_state)    : %.4f | %% of Total Variance: %6.2f%%\n", sigma_state_sq, vpc_state * 100))
cat(sprintf("3. Latent order-level residual variance (sigma^2_residual) : %.4f | %% of Total Variance: %6.2f%%\n", sigma_resid_sq, vpc_residual * 100))
cat("========================================================================================\n\n")


================ Multilevel Complaint Risk Variance Decomposition (VPC) ================
1. Individual seller effect variance  (sigma^2_seller)   : 0.2546 | % of Total Variance:   7.18%
2. Seller state geographical variance (sigma^2_state)    : 0.0000 | % of Total Variance:   0.00%
3. Latent order-level residual variance (sigma^2_residual) : 3.2899 | % of Total Variance:  92.82%



The VPC decomposition shows that individual seller explains more variance to residual variance than the seller state variance. And seller state variance is nearly zero. This means the variance of this high level cluster is already explained by the individual seller lower cluster. This implies: 

The complain review occurrences comes from "seller-level individual heterogeneity", not state-level perspective. That means the complain review depends on the quality control of product and service provided by seller but not the seller state geographical disadvantage. 

But almost variance still comes from order level which contains unobserved specific characteristics such as customer purchase behavior or preferences. 

In [29]:
# Extract fixed-effects Conditional Odds Ratios (Conditional OR)
fixef_glmm <- summary(glmm_complaint_multilevel)$coefficients
or_glmm_df <- data.frame(
  Covariate   = rownames(fixef_glmm),
  Estimate    = fixef_glmm[, "Estimate"],
  Cond_OR     = exp(fixef_glmm[, "Estimate"]),
  CI_2.5pct   = exp(fixef_glmm[, "Estimate"] - 1.96 * fixef_glmm[, "Std. Error"]),
  CI_97.5pct  = exp(fixef_glmm[, "Estimate"] + 1.96 * fixef_glmm[, "Std. Error"]),
  P_value     = fixef_glmm[, "Pr(>|z|)"]
)
print(round(or_glmm_df[, -1], 4), row.names = or_glmm_df$Covariate)

                      Estimate Cond_OR CI_2.5pct CI_97.5pct P_value
(Intercept)            -2.6019  0.0741    0.0704     0.0781  0.0000
delivery_delay_days_z   0.8138  2.2564    2.2067     2.3073  0.0000
freight_ratio_z         0.0161  1.0162    0.9905     1.0427  0.2191
is_cross_state1         0.4766  1.6105    1.5355     1.6893  0.0000


The table above shows that the p-value of 'delivery_delay_days_z', 'is_cross_state1' are both smaller than 0.05 which indicate their coefficient statistically significant contribution to Multilevel Binomial GLMM at 95% confidence level. By this, we conclude:

Conditional on seller himself/herself and seller state, the odd of extreme negative review is dramatically increased to 2.26 times of the original with 1 unit/1 standard deviation increase in delivery delay days. This means delivery delay day is a high risk factor of receive extreme negative review and the later the customer receive the product, the more chance of being complained and receiving negative review. 

Conditional on seller himself/herself and seller state, the odd of extreme negative review increase 61.1% of its original due to cross state transaction compare to same state transaction. This means cross state trade is also a high risk factor of receive extreme negative review. 